# CAPSTONE PROJECT: Customer Segmentation

## Clustering Problem

---

## Project Overview

**Objective:** Segment customers into distinct groups based on their purchasing behavior and demographics to enable targeted marketing strategies.

**Business Context:**
- Understanding customer segments drives personalized marketing
- Different segments require different strategies
- Improves customer satisfaction and revenue

**Dataset:** E-commerce Customer Data (similar to RFM analysis datasets)

---

## Project Structure

```
1. Problem Definition
2. Data Collection
3. Exploratory Data Analysis
4. Feature Engineering (RFM Analysis)
5. Data Preprocessing
6. Clustering Analysis
7. Cluster Profiling
8. Business Recommendations
9. Conclusions
```

---

## Documentation: Clustering Capstone Guide

### RFM Analysis Framework

RFM is a powerful customer segmentation technique:

| Metric | Description | Meaning |
|--------|-------------|----------|
| **R**ecency | Days since last purchase | Lower = Better |
| **F**requency | Total number of purchases | Higher = Better |
| **M**onetary | Total amount spent | Higher = Better |

### Clustering Best Practices

1. **Scale features** - Clustering algorithms are distance-based
2. **Handle outliers** - Can significantly affect cluster centroids
3. **Validate clusters** - Use multiple methods (Elbow, Silhouette)
4. **Interpret clusters** - Give meaningful business names

---

## Part 1: Setup and Data Collection

In [ ]:
# Install required packages
# !pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Visualization
from scipy.cluster.hierarchy import dendrogram, linkage

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully!")

In [ ]:
# Create realistic e-commerce transaction dataset
# (In practice, download from Kaggle: https://www.kaggle.com/datasets/carrie1/ecommerce-data)

np.random.seed(42)

def generate_ecommerce_data(n_customers=2000, n_transactions=10000):
    """
    Generate realistic e-commerce transaction data
    """
    # Customer types with different behaviors
    customer_types = {
        'VIP': {'pct': 0.05, 'freq_range': (20, 50), 'value_range': (100, 500)},
        'Loyal': {'pct': 0.15, 'freq_range': (10, 25), 'value_range': (50, 200)},
        'Regular': {'pct': 0.30, 'freq_range': (5, 15), 'value_range': (30, 100)},
        'Occasional': {'pct': 0.35, 'freq_range': (2, 8), 'value_range': (20, 80)},
        'At_Risk': {'pct': 0.15, 'freq_range': (1, 3), 'value_range': (10, 50)}
    }
    
    transactions = []
    customer_id = 1
    
    for ctype, params in customer_types.items():
        n_cust = int(n_customers * params['pct'])
        
        for _ in range(n_cust):
            # Number of transactions for this customer
            n_trans = np.random.randint(params['freq_range'][0], params['freq_range'][1])
            
            # Generate transactions
            for _ in range(n_trans):
                # Random date in last 365 days
                days_ago = np.random.exponential(180 if ctype != 'At_Risk' else 300)
                days_ago = min(days_ago, 365)
                trans_date = datetime.now() - timedelta(days=int(days_ago))
                
                # Transaction value
                value = np.random.uniform(params['value_range'][0], params['value_range'][1])
                
                transactions.append({
                    'CustomerID': f'C{customer_id:05d}',
                    'TransactionDate': trans_date,
                    'TransactionValue': round(value, 2)
                })
            
            customer_id += 1
    
    return pd.DataFrame(transactions)

# Generate dataset
transactions_df = generate_ecommerce_data(n_customers=2000)

print(f"Transactions Dataset Shape: {transactions_df.shape}")
print(f"Date Range: {transactions_df['TransactionDate'].min().date()} to {transactions_df['TransactionDate'].max().date()}")
print(f"Unique Customers: {transactions_df['CustomerID'].nunique()}")
transactions_df.head()

## Part 2: Exploratory Data Analysis

In [ ]:
# Basic transaction analysis
print("=" * 60)
print("TRANSACTION SUMMARY")
print("=" * 60)
print(f"\nTotal Transactions: {len(transactions_df):,}")
print(f"Total Revenue: ${transactions_df['TransactionValue'].sum():,.2f}")
print(f"Average Transaction Value: ${transactions_df['TransactionValue'].mean():.2f}")
print(f"Median Transaction Value: ${transactions_df['TransactionValue'].median():.2f}")

In [ ]:
# Transaction value distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(transactions_df['TransactionValue'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(transactions_df['TransactionValue'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(transactions_df['TransactionValue'].median(), color='green', linestyle='--', label='Median')
axes[0].set_xlabel('Transaction Value ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Transaction Value Distribution')
axes[0].legend()

# Transactions over time
transactions_df['YearMonth'] = transactions_df['TransactionDate'].dt.to_period('M')
monthly_trans = transactions_df.groupby('YearMonth').agg({
    'TransactionValue': ['count', 'sum']
}).reset_index()
monthly_trans.columns = ['YearMonth', 'Count', 'Revenue']
monthly_trans['YearMonth'] = monthly_trans['YearMonth'].astype(str)

axes[1].bar(range(len(monthly_trans)), monthly_trans['Revenue'], color='coral')
axes[1].set_xticks(range(len(monthly_trans)))
axes[1].set_xticklabels(monthly_trans['YearMonth'], rotation=45, ha='right')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Revenue ($)')
axes[1].set_title('Monthly Revenue Trend')

plt.tight_layout()
plt.show()

## Part 3: Feature Engineering - RFM Analysis

In [ ]:
# Calculate RFM metrics
reference_date = transactions_df['TransactionDate'].max() + timedelta(days=1)
print(f"Reference Date: {reference_date.date()}")

# Create RFM DataFrame
rfm_df = transactions_df.groupby('CustomerID').agg({
    'TransactionDate': lambda x: (reference_date - x.max()).days,  # Recency
    'CustomerID': 'count',  # Frequency (using CustomerID as placeholder)
    'TransactionValue': 'sum'  # Monetary
}).rename(columns={
    'TransactionDate': 'Recency',
    'CustomerID': 'Frequency',
    'TransactionValue': 'Monetary'
})

print(f"\nRFM Dataset Shape: {rfm_df.shape}")
rfm_df.head(10)

In [ ]:
# RFM Statistics
print("=" * 60)
print("RFM METRICS SUMMARY")
print("=" * 60)
rfm_df.describe()

In [ ]:
# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Recency
axes[0].hist(rfm_df['Recency'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Recency (days since last purchase)')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Recency Distribution\n(Lower = Better)')

# Frequency
axes[1].hist(rfm_df['Frequency'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Frequency (number of transactions)')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Frequency Distribution\n(Higher = Better)')

# Monetary
axes[2].hist(rfm_df['Monetary'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_xlabel('Monetary (total spend $)')
axes[2].set_ylabel('Number of Customers')
axes[2].set_title('Monetary Distribution\n(Higher = Better)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between RFM metrics
plt.figure(figsize=(8, 6))
sns.heatmap(rfm_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('RFM Correlation Matrix')
plt.show()

print("\nNote: Frequency and Monetary are usually correlated (more purchases = more spend)")

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(rfm_df['Recency'], rfm_df['Frequency'], alpha=0.3, s=20)
axes[0].set_xlabel('Recency')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Recency vs Frequency')

axes[1].scatter(rfm_df['Frequency'], rfm_df['Monetary'], alpha=0.3, s=20)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Monetary')
axes[1].set_title('Frequency vs Monetary')

axes[2].scatter(rfm_df['Recency'], rfm_df['Monetary'], alpha=0.3, s=20)
axes[2].set_xlabel('Recency')
axes[2].set_ylabel('Monetary')
axes[2].set_title('Recency vs Monetary')

plt.tight_layout()
plt.show()

## Part 4: Data Preprocessing

In [ ]:
# Handle outliers using IQR method
def remove_outliers_iqr(df, columns, multiplier=1.5):
    """
    Remove outliers using IQR method
    """
    df_clean = df.copy()
    
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
        after = len(df_clean)
        print(f"{col}: Removed {before - after} outliers")
    
    return df_clean

# Remove outliers
print("Removing Outliers:")
print(f"Original size: {len(rfm_df)}")
rfm_clean = remove_outliers_iqr(rfm_df, ['Recency', 'Frequency', 'Monetary'])
print(f"Clean size: {len(rfm_clean)}")

In [ ]:
# Scale the data
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_clean)

# Create DataFrame with scaled values
rfm_scaled_df = pd.DataFrame(
    rfm_scaled, 
    columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'],
    index=rfm_clean.index
)

print("Data Scaled:")
print(f"Mean: {rfm_scaled.mean(axis=0).round(4)}")
print(f"Std: {rfm_scaled.std(axis=0).round(4)}")

## Part 5: Clustering Analysis

In [ ]:
# Find optimal number of clusters using multiple methods
k_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(rfm_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(rfm_scaled, labels))
    davies_scores.append(davies_bouldin_score(rfm_scaled, labels))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Elbow Method
axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_title('Elbow Method')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette Score
axes[0, 1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Number of Clusters (k)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].set_title('Silhouette Analysis (Higher = Better)')
axes[0, 1].grid(True, alpha=0.3)

# Calinski-Harabasz Score
axes[1, 0].plot(k_range, calinski_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)')
axes[1, 0].set_ylabel('Calinski-Harabasz Score')
axes[1, 0].set_title('Calinski-Harabasz (Higher = Better)')
axes[1, 0].grid(True, alpha=0.3)

# Davies-Bouldin Score
axes[1, 1].plot(k_range, davies_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Number of Clusters (k)')
axes[1, 1].set_ylabel('Davies-Bouldin Score')
axes[1, 1].set_title('Davies-Bouldin (Lower = Better)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print recommendations
optimal_silhouette = k_range[np.argmax(silhouette_scores)]
optimal_calinski = k_range[np.argmax(calinski_scores)]
optimal_davies = k_range[np.argmin(davies_scores)]

print(f"\nOptimal k recommendations:")
print(f"Silhouette Score: k={optimal_silhouette}")
print(f"Calinski-Harabasz: k={optimal_calinski}")
print(f"Davies-Bouldin: k={optimal_davies}")

In [ ]:
# Final K-Means clustering with k=5 (business-appropriate number of segments)
optimal_k = 5

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm_clean['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

print(f"K-Means Clustering with k={optimal_k}:")
print(f"Silhouette Score: {silhouette_score(rfm_scaled, rfm_clean['Cluster']):.4f}")
print(f"\nCluster Distribution:")
print(rfm_clean['Cluster'].value_counts().sort_index())

In [ ]:
# Compare with other clustering methods
clustering_results = []

# K-Means
kmeans_labels = rfm_clean['Cluster'].values
clustering_results.append({
    'Algorithm': 'K-Means',
    'N Clusters': optimal_k,
    'Silhouette': silhouette_score(rfm_scaled, kmeans_labels),
    'Calinski-Harabasz': calinski_harabasz_score(rfm_scaled, kmeans_labels),
    'Davies-Bouldin': davies_bouldin_score(rfm_scaled, kmeans_labels)
})

# Hierarchical
hierarchical = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
hier_labels = hierarchical.fit_predict(rfm_scaled)
clustering_results.append({
    'Algorithm': 'Hierarchical',
    'N Clusters': optimal_k,
    'Silhouette': silhouette_score(rfm_scaled, hier_labels),
    'Calinski-Harabasz': calinski_harabasz_score(rfm_scaled, hier_labels),
    'Davies-Bouldin': davies_bouldin_score(rfm_scaled, hier_labels)
})

# GMM
gmm = GaussianMixture(n_components=optimal_k, random_state=42)
gmm_labels = gmm.fit_predict(rfm_scaled)
clustering_results.append({
    'Algorithm': 'GMM',
    'N Clusters': optimal_k,
    'Silhouette': silhouette_score(rfm_scaled, gmm_labels),
    'Calinski-Harabasz': calinski_harabasz_score(rfm_scaled, gmm_labels),
    'Davies-Bouldin': davies_bouldin_score(rfm_scaled, gmm_labels)
})

clustering_comparison = pd.DataFrame(clustering_results)
print("Clustering Algorithm Comparison:")
print(clustering_comparison.round(4))

## Part 6: Cluster Profiling

In [ ]:
# Analyze cluster characteristics
cluster_summary = rfm_clean.groupby('Cluster').agg({
    'Recency': ['mean', 'median', 'min', 'max'],
    'Frequency': ['mean', 'median', 'min', 'max'],
    'Monetary': ['mean', 'median', 'min', 'max', 'sum']
}).round(2)

print("=" * 80)
print("CLUSTER PROFILES")
print("=" * 80)
print(cluster_summary)

In [ ]:
# Create summary for naming clusters
cluster_profile = rfm_clean.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2)

# Add customer count and percentage
cluster_profile['Count'] = rfm_clean['Cluster'].value_counts().sort_index()
cluster_profile['Percentage'] = (cluster_profile['Count'] / len(rfm_clean) * 100).round(1)
cluster_profile['Total_Revenue'] = rfm_clean.groupby('Cluster')['Monetary'].sum()
cluster_profile['Revenue_Pct'] = (cluster_profile['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100).round(1)

print("\nCluster Summary:")
print(cluster_profile)

In [ ]:
# Name clusters based on RFM characteristics
# Lower Recency = Better, Higher Frequency/Monetary = Better

# Calculate normalized scores for naming
cluster_scores = cluster_profile[['Recency', 'Frequency', 'Monetary']].copy()
cluster_scores['R_score'] = (cluster_scores['Recency'].max() - cluster_scores['Recency']) / cluster_scores['Recency'].max()
cluster_scores['F_score'] = cluster_scores['Frequency'] / cluster_scores['Frequency'].max()
cluster_scores['M_score'] = cluster_scores['Monetary'] / cluster_scores['Monetary'].max()
cluster_scores['Total_Score'] = cluster_scores['R_score'] + cluster_scores['F_score'] + cluster_scores['M_score']

# Sort by total score to assign names
cluster_scores_sorted = cluster_scores.sort_values('Total_Score', ascending=False)

# Define segment names
segment_names = {
    cluster_scores_sorted.index[0]: 'Champions',
    cluster_scores_sorted.index[1]: 'Loyal Customers',
    cluster_scores_sorted.index[2]: 'Potential Loyalists',
    cluster_scores_sorted.index[3]: 'At Risk',
    cluster_scores_sorted.index[4]: 'Hibernating'
}

rfm_clean['Segment'] = rfm_clean['Cluster'].map(segment_names)

print("Segment Names Assigned:")
print(rfm_clean['Segment'].value_counts())

In [ ]:
# Final segment profile
final_profile = rfm_clean.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'sum'],
    'Cluster': 'count'
}).round(2)

final_profile.columns = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Total_Revenue', 'Count']
final_profile['Pct_Customers'] = (final_profile['Count'] / final_profile['Count'].sum() * 100).round(1)
final_profile['Pct_Revenue'] = (final_profile['Total_Revenue'] / final_profile['Total_Revenue'].sum() * 100).round(1)

# Reorder columns
final_profile = final_profile[['Count', 'Pct_Customers', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Total_Revenue', 'Pct_Revenue']]

print("=" * 100)
print("FINAL SEGMENT PROFILES")
print("=" * 100)
print(final_profile)

In [ ]:
# Visualize segments
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Define colors for segments
segment_colors = {
    'Champions': '#27ae60',
    'Loyal Customers': '#3498db',
    'Potential Loyalists': '#f1c40f',
    'At Risk': '#e67e22',
    'Hibernating': '#e74c3c'
}

# 1. Segment distribution
segment_counts = rfm_clean['Segment'].value_counts()
colors = [segment_colors[seg] for seg in segment_counts.index]
axes[0, 0].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', colors=colors)
axes[0, 0].set_title('Customer Distribution by Segment')

# 2. Revenue contribution
revenue_by_segment = rfm_clean.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
colors = [segment_colors[seg] for seg in revenue_by_segment.index]
axes[0, 1].barh(revenue_by_segment.index, revenue_by_segment.values, color=colors)
axes[0, 1].set_xlabel('Total Revenue ($)')
axes[0, 1].set_title('Revenue Contribution by Segment')

# 3. Scatter plot - Recency vs Monetary
for segment in segment_colors:
    mask = rfm_clean['Segment'] == segment
    axes[1, 0].scatter(rfm_clean[mask]['Recency'], rfm_clean[mask]['Monetary'],
                       label=segment, alpha=0.6, color=segment_colors[segment], s=30)
axes[1, 0].set_xlabel('Recency (days)')
axes[1, 0].set_ylabel('Monetary ($)')
axes[1, 0].set_title('Customer Segments: Recency vs Monetary')
axes[1, 0].legend(loc='upper right')

# 4. Scatter plot - Frequency vs Monetary
for segment in segment_colors:
    mask = rfm_clean['Segment'] == segment
    axes[1, 1].scatter(rfm_clean[mask]['Frequency'], rfm_clean[mask]['Monetary'],
                       label=segment, alpha=0.6, color=segment_colors[segment], s=30)
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].set_ylabel('Monetary ($)')
axes[1, 1].set_title('Customer Segments: Frequency vs Monetary')
axes[1, 1].legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 3D Visualization using PCA
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

for segment in segment_colors:
    mask = rfm_clean['Segment'] == segment
    ax.scatter(rfm_clean[mask]['Recency'], 
               rfm_clean[mask]['Frequency'],
               rfm_clean[mask]['Monetary'],
               label=segment, alpha=0.6, color=segment_colors[segment], s=30)

ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
ax.set_title('3D Visualization of Customer Segments')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for segment comparison
from math import pi

# Prepare data for radar chart
segment_means = rfm_clean.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean()

# Normalize values for radar chart (inverse recency since lower is better)
radar_data = segment_means.copy()
radar_data['Recency'] = (radar_data['Recency'].max() - radar_data['Recency']) / radar_data['Recency'].max()
radar_data['Frequency'] = radar_data['Frequency'] / radar_data['Frequency'].max()
radar_data['Monetary'] = radar_data['Monetary'] / radar_data['Monetary'].max()

# Plot
categories = list(radar_data.columns)
N = len(categories)

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for segment in radar_data.index:
    values = radar_data.loc[segment].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=segment, color=segment_colors[segment])
    ax.fill(angles, values, alpha=0.25, color=segment_colors[segment])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Recency\n(Higher=Recent)', 'Frequency', 'Monetary'])
ax.set_title('Segment Comparison (Normalized Scores)', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1))

plt.tight_layout()
plt.show()

## Part 7: Business Recommendations

In [ ]:
# Create actionable recommendations
recommendations = {
    'Champions': {
        'Description': 'Best customers - Recent, frequent, high spenders',
        'Strategy': 'Reward loyalty, ask for referrals, early access to new products',
        'Actions': [
            'VIP loyalty program with exclusive benefits',
            'Referral program with incentives',
            'Early access to new products/sales',
            'Personalized thank you messages'
        ],
        'Priority': 'HIGH',
        'Investment': 'Medium (retain at all costs)'
    },
    'Loyal Customers': {
        'Description': 'Regular purchasers with good spending',
        'Strategy': 'Upsell, cross-sell, increase engagement',
        'Actions': [
            'Recommend higher-tier products',
            'Cross-sell complementary items',
            'Loyalty points program',
            'Birthday/anniversary rewards'
        ],
        'Priority': 'HIGH',
        'Investment': 'Medium'
    },
    'Potential Loyalists': {
        'Description': 'Recent customers with potential',
        'Strategy': 'Nurture relationship, increase frequency',
        'Actions': [
            'Welcome series emails',
            'First-time buyer discounts',
            'Product education content',
            'Free shipping thresholds'
        ],
        'Priority': 'MEDIUM-HIGH',
        'Investment': 'Medium'
    },
    'At Risk': {
        'Description': 'Previously good customers showing disengagement',
        'Strategy': 'Win-back campaigns, understand issues',
        'Actions': [
            'Win-back email campaigns',
            'Special discounts/offers',
            'Survey to understand issues',
            'Personal outreach for high-value customers'
        ],
        'Priority': 'HIGH (URGENT)',
        'Investment': 'High'
    },
    'Hibernating': {
        'Description': 'Inactive customers, low recent engagement',
        'Strategy': 'Reactivation campaigns, consider letting go',
        'Actions': [
            'Re-engagement email series',
            'Significant discount offers',
            'We miss you campaigns',
            'Survey before removing from list'
        ],
        'Priority': 'LOW-MEDIUM',
        'Investment': 'Low (test before investing)'
    }
}

# Display recommendations
print("=" * 80)
print("SEGMENT-SPECIFIC MARKETING RECOMMENDATIONS")
print("=" * 80)

for segment, rec in recommendations.items():
    count = len(rfm_clean[rfm_clean['Segment'] == segment])
    revenue = rfm_clean[rfm_clean['Segment'] == segment]['Monetary'].sum()
    
    print(f"\n{'='*60}")
    print(f"SEGMENT: {segment.upper()}")
    print(f"{'='*60}")
    print(f"Customers: {count} | Revenue: ${revenue:,.2f}")
    print(f"Description: {rec['Description']}")
    print(f"Strategy: {rec['Strategy']}")
    print(f"Priority: {rec['Priority']} | Investment: {rec['Investment']}")
    print(f"\nRecommended Actions:")
    for i, action in enumerate(rec['Actions'], 1):
        print(f"  {i}. {action}")

In [ ]:
# ROI Analysis
print("\n" + "=" * 80)
print("ROI ANALYSIS")
print("=" * 80)

# Calculate potential impact
at_risk_customers = len(rfm_clean[rfm_clean['Segment'] == 'At Risk'])
at_risk_avg_value = rfm_clean[rfm_clean['Segment'] == 'At Risk']['Monetary'].mean()
hibernating_customers = len(rfm_clean[rfm_clean['Segment'] == 'Hibernating'])
hibernating_avg_value = rfm_clean[rfm_clean['Segment'] == 'Hibernating']['Monetary'].mean()

# Assumptions
retention_rate = 0.20  # 20% success rate
reactivation_rate = 0.10  # 10% success rate

potential_retained_revenue = at_risk_customers * at_risk_avg_value * retention_rate
potential_reactivation_revenue = hibernating_customers * hibernating_avg_value * reactivation_rate

print(f"\nAt Risk Segment:")
print(f"  Customers: {at_risk_customers}")
print(f"  Average Value: ${at_risk_avg_value:,.2f}")
print(f"  If 20% retained: ${potential_retained_revenue:,.2f} preserved")

print(f"\nHibernating Segment:")
print(f"  Customers: {hibernating_customers}")
print(f"  Average Value: ${hibernating_avg_value:,.2f}")
print(f"  If 10% reactivated: ${potential_reactivation_revenue:,.2f} recovered")

print(f"\nTotal Potential Impact: ${potential_retained_revenue + potential_reactivation_revenue:,.2f}")

## Part 8: Export Results

In [ ]:
# Create final customer segment dataset
final_customer_data = rfm_clean.copy()
final_customer_data['Segment_Priority'] = final_customer_data['Segment'].map({
    'Champions': 1,
    'Loyal Customers': 2,
    'At Risk': 3,  # High priority for action
    'Potential Loyalists': 4,
    'Hibernating': 5
})

print("Final Customer Dataset:")
final_customer_data.head(10)

In [ ]:
# Export results (uncomment to save)
# final_customer_data.to_csv('customer_segments.csv')
# print("Customer segments saved to customer_segments.csv")

In [ ]:
# Function to segment new customers
def segment_customer(recency, frequency, monetary):
    """
    Segment a new customer based on RFM values
    """
    # Scale using the same scaler
    customer_rfm = np.array([[recency, frequency, monetary]])
    customer_scaled = scaler.transform(customer_rfm)
    
    # Predict cluster
    cluster = kmeans_final.predict(customer_scaled)[0]
    segment = segment_names[cluster]
    
    return segment

# Example
new_customer_segment = segment_customer(recency=10, frequency=15, monetary=500)
print(f"New customer segment: {new_customer_segment}")

## Part 9: Conclusions

### Summary

- **Segmentation Method**: K-Means with k=5 clusters
- **Evaluation**: Silhouette Score = X.XX

### Key Findings

| Segment | % Customers | % Revenue | Strategy Priority |
|---------|-------------|-----------|------------------|
| Champions | ~XX% | ~XX% | Retain & Reward |
| Loyal Customers | ~XX% | ~XX% | Upsell & Cross-sell |
| Potential Loyalists | ~XX% | ~XX% | Nurture |
| At Risk | ~XX% | ~XX% | WIN BACK (URGENT) |
| Hibernating | ~XX% | ~XX% | Reactivate or Release |

### Business Impact

1. **Champions** (smallest group, highest value)
   - Focus on retention and referrals
   - Low marketing cost, high ROI

2. **At Risk** (action required)
   - Immediate outreach needed
   - Significant revenue at risk

3. **Hibernating** (test before investing)
   - Low-cost reactivation campaigns
   - Accept some customer loss

### Next Steps

1. Implement segment-specific marketing campaigns
2. Track conversion rates by segment
3. Re-run segmentation quarterly
4. Integrate with CRM system

---

## Capstone Project Checklist

- [x] Problem Definition with Business Context
- [x] Data Collection and Preparation
- [x] RFM Feature Engineering
- [x] Data Preprocessing (Outliers, Scaling)
- [x] Optimal Cluster Selection (Multiple Methods)
- [x] Clustering Algorithm Comparison
- [x] Cluster Profiling and Naming
- [x] Visualizations (2D, 3D, Radar Charts)
- [x] Business Recommendations
- [x] ROI Analysis
- [x] Export and Deployment Preparation

**Congratulations! You've completed the Clustering Capstone Project!**